In [0]:
train_df = spark.table("insurance_project.train_insurance")
test_df = spark.table("insurance_project.test_insurance")

print("Train Rows:", train_df.count())
print("Test Rows:", test_df.count())

Train Rows: 1081
Test Rows: 257


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "age",
        "bmi",
        "children",
        "smoker_flag"
    ],
    outputCol="features"
)

train_data = assembler.transform(train_df)
test_data = assembler.transform(test_df)

In [0]:
display(
    train_data.select(
        "features",
        "charges"
    )
)

features,charges
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""20.79"",""0.0"",""0.0""]}",1607.5101
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""21.66"",""0.0"",""1.0""]}",14283.4594
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""24.09"",""1.0"",""0.0""]}",2201.0971
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""25.08"",""0.0"",""0.0""]}",2196.4732
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""26.315"",""0.0"",""0.0""]}",2198.18985
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""26.73"",""0.0"",""0.0""]}",1615.7667
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""27.28"",""3.0"",""1.0""]}",18223.4512
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""28.215"",""0.0"",""0.0""]}",2200.83085
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""30.115"",""0.0"",""0.0""]}",2203.47185
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""30.115"",""0.0"",""0.0""]}",21344.8467


In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="charges",
    numTrees=100,
    maxDepth=5,
    seed=42
)

In [0]:
rf_model = rf.fit(train_data)

In [0]:
print(rf_model)

RandomForestRegressionModel: uid=RandomForestRegressor_2e59062902c5, numTrees=100, numFeatures=4


In [0]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="charges",
    numTrees=100
)
rf_model = rf.fit(train_data)
print(rf_model)

RandomForestRegressionModel: uid=RandomForestRegressor_c4e8aaf4b094, numTrees=100, numFeatures=4


In [0]:
rf_predictions = rf_model.transform(test_data)

In [0]:
display(
    rf_predictions.select(
        "charges",
        "prediction"
    )
)

charges,prediction
7323.734819,3787.1897005457436
2203.73595,3952.780099602624
1622.1885,4102.147385367873
36149.4835,36340.38354234026
1631.6683,3991.657659356952
1631.8212,3991.657659356952
14133.03775,3991.657659356952
1633.0444,3910.1202330924652
1634.5734,3910.1202330924652
12829.4551,17452.744445342207


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

r2_eval = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="r2"
)

r2_rf = r2_eval.evaluate(
    rf_predictions
)

print("Random Forest R2:", r2_rf)

Random Forest R2: 0.8760504408233597


In [0]:
rmse_eval = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="rmse"
)
rmse_rf = rmse_eval.evaluate(
    rf_predictions
)

print("RMSE:", rmse_rf)

RMSE: 4060.455884135414


In [0]:
mae_eval = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="mae"
)
mae_rf = mae_eval.evaluate(
    rf_predictions
)

print("MAE:", mae_rf)

MAE: 2430.488596631859


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE
rmse = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(rf_predictions)

# MAE
mae = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="mae"
).evaluate(rf_predictions)

# R2
r2 = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="r2"
).evaluate(rf_predictions)

print("RMSE =", rmse)
print("MAE =", mae)
print("R² =", r2)

RMSE = 4060.455884135414
MAE = 2430.488596631859
R² = 0.8760504408233597


In [0]:
print("Linear Regression R2:", 0.7504)
print("Random Forest R2:", r2_rf)

Linear Regression R2: 0.7504
Random Forest R2: 0.8760504408233597


In [0]:
importance = rf_model.featureImportances

feature_names = [
    "age",
    "bmi",
    "children",
    "smoker_flag"
]

for name, score in zip(feature_names, importance):
    print(name, score)

age 0.12151444253321037
bmi 0.14016139007566897
children 0.011578051986823414
smoker_flag 0.7267461154042972


# Day 17 - Gradient Boosting Regressor

In [0]:
from pyspark.ml.regression import GBTRegressor
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="charges",
    maxIter=100,
    maxDepth=5,
    seed=42
)
gbt_model = gbt.fit(train_data)
gbt_predictions = gbt_model.transform(test_data)

display(
    gbt_predictions.select(
        "charges",
        "prediction"
    )
)


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

r2_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="r2"
).evaluate(gbt_predictions)

print("Gradient Boosting R2:", r2_gbt)

In [0]:
rmse_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(gbt_predictions)

print("Gradient Boosting RMSE:", rmse_gbt)

In [0]:
mae_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="mae"
).evaluate(gbt_predictions)

print("Gradient Boosting MAE:", mae_gbt)

In [0]:
print("Model Comparison")
print("----------------")
print("Linear Regression R2 :", 0.7504)
print("Random Forest R2     :", 0.8727)
print("Gradient Boosting R2 :", r2_gbt)

In [0]:
feature_names = [
    "age",
    "bmi",
    "children",
    "smoker_flag",
    "sex_index",
    "region_index",
    "bmi_index",
    "age_group_index"
]

for name, score in zip(feature_names, gbt_model.featureImportances):
    print(f"{name}: {score}")

In [0]:
train_df = spark.table("insurance_project.train_insurance")
test_df = spark.table("insurance_project.test_insurance")

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "age",
        "bmi",
        "children",
        "smoker_flag",
        "sex_index",
        "region_index",
        "bmi_index",
        "age_group_index"
    ],
    outputCol="features"
)

train_data = assembler.transform(train_df)
test_data = assembler.transform(test_df)

print("Feature vectors created")

In [0]:
print(train_data.count())
print(test_data.count())

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="charges",
    maxIter=100,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_data)

print("GBT Model Trained")

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="charges",
    maxIter=100,
    maxDepth=5,
    seed=42
)

print("GBT Model Created")

In [0]:
gbt_predictions = gbt_model.transform(test_data)

display(
    gbt_predictions.select(
        "charges",
        "prediction"
    )
)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

r2_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="r2"
).evaluate(gbt_predictions)

rmse_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(gbt_predictions)

mae_gbt = RegressionEvaluator(
    labelCol="charges",
    predictionCol="prediction",
    metricName="mae"
).evaluate(gbt_predictions)

print("R2 :", r2_gbt)
print("RMSE :", rmse_gbt)
print("MAE :", mae_gbt)

In [0]:
r2_rf = 0.8727228514327479
rmse_rf = 4114.599051379543
mae_rf = 2597.4317232836697

# Day 17 Conclusion

Linear Regression R² : 0.7504

Random Forest R² : 0.8727

Gradient Boosting R² : 0.8063

Best Model : Gradient Boosting R²

In [0]:
import mlflow
mlflow.set_experiment(
    "/Users/gomathilakshmigandhi@gmail.com/Medical_Insurance_Prediction_Experiment"
)

In [0]:
import mlflow

with mlflow.start_run(run_name="Random_Forest_Model"):

    mlflow.log_param("Model", "Random Forest")

    mlflow.log_metric("R2", r2_rf)
    mlflow.log_metric("RMSE", rmse_rf)
    mlflow.log_metric("MAE", mae_rf)

    print("Run Logged Successfully")

In [0]:
import mlflow

mlflow.end_run()

print("Run Closed")

In [0]:
import mlflow

with mlflow.start_run(run_name="Gradient_Boosting_Model"):

    mlflow.log_param("Model", "Gradient Boosting")
    mlflow.log_metric("R2", r2_gbt)
    mlflow.log_metric("RMSE", rmse_gbt)
    mlflow.log_metric("MAE", mae_gbt)

print("Gradient Boosting Logged Successfully")

In [0]:
import mlflow

mlflow.end_run()

print("Run Closed")

In [0]:
import mlflow

with mlflow.start_run(run_name="Linear_Regression_Model"):
    mlflow.log_param("Model", "Linear Regression")
    mlflow.log_metric("R2", 0.7504)

print("Linear Regression Logged")

In [0]:
print("R2 :", r2_gbt)
print("RMSE :", rmse_gbt)
print("MAE :", mae_gbt)

In [0]:
import mlflow

with mlflow.start_run(run_name="Random_Forest_Model_Final"):

    mlflow.log_param("Model", "Random Forest")

    mlflow.log_metric("R2", 0.8727228514327479)
    mlflow.log_metric("RMSE", 4114.599051379543)
    mlflow.log_metric("MAE", 2597.4317232836697)

print("Metrics Logged Successfully")

In [0]:
%sql
SHOW VOLUMES;

In [0]:
%sql
USE insurance_project;

In [0]:
%sql
CREATE VOLUME insurance_ml_volume;

In [0]:
%sql
DESCRIBE VOLUME insurance_ml_volume;

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/insurance_project/insurance_ml_volume/tmp"
)

print("Temp folder created")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/insurance_project/insurance_ml_volume/"
    )
)

In [0]:
train_df = spark.table("insurance_project.train_insurance")
test_df = spark.table("insurance_project.test_insurance")

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "age",
        "bmi",
        "children",
        "smoker_flag",
        "sex_index",
        "region_index",
        "bmi_index",
        "age_group_index"
    ],
    outputCol="features"
)

train_data = assembler.transform(train_df)
test_data = assembler.transform(test_df)

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="charges",
    numTrees=100,
    maxDepth=5,
    seed=42
)

rf_model = rf.fit(train_data)

print("Random Forest Model Recreated")

In [0]:
print(rf_model)

In [0]:
import mlflow

mlflow.set_experiment(
    "/Users/gomathilakshmigandhi@gmail.com/Medical_Insurance_Prediction_Experiment"
)

In [0]:
with mlflow.start_run(run_name="Random_Forest_Model_Registry"):

    mlflow.log_param("Model", "Random Forest")

    mlflow.log_metric("R2", 0.8727228514327479)
    mlflow.log_metric("RMSE", 4114.599051379543)
    mlflow.log_metric("MAE", 2597.4317232836697)

print("Run Logged")

In [0]:
sample_input = train_data.limit(5).toPandas()

sample_input

In [0]:
from mlflow.models.signature import infer_signature

sample_output = rf_model.transform(train_data.limit(5)) \
                        .select("prediction") \
                        .toPandas()

signature = infer_signature(
    sample_input_clean,
    sample_output
)

In [0]:
sample_input_clean = sample_input.drop(columns=["features"])

print(sample_input_clean.columns)

In [0]:
print(sample_input.columns)

In [0]:
import mlflow
import mlflow.spark

with mlflow.start_run(run_name="Random_Forest_Model_With_Signature"):

    mlflow.spark.log_model(
        spark_model=rf_model,
        artifact_path="random_forest_model",
        signature=signature,
        input_example=sample_input_clean,
        dfs_tmpdir="/Volumes/workspace/insurance_project/insurance_ml_volume/tmp"
    )

print("Model Logged With Signature")

In [0]:
sample_input_clean = train_df.select(
    "age",
    "sex",
    "bmi",
    "children",
    "smoker",
    "region"
).limit(5).toPandas()

sample_input_clean

In [0]:
from mlflow.models.signature import infer_signature

sample_output = rf_model.transform(train_data.limit(5)) \
    .select("prediction") \
    .toPandas()

signature = infer_signature(sample_input_clean, sample_output)

In [0]:
import mlflow
import mlflow.spark

with mlflow.start_run(run_name="Random_Forest_Model_With_Signature"):

    mlflow.log_param("Model", "Random Forest")

    mlflow.log_metric("R2", 0.8727228514327479)
    mlflow.log_metric("RMSE", 4114.599051379543)
    mlflow.log_metric("MAE", 2597.4317232836697)

    mlflow.spark.log_model(
        spark_model=rf_model,
        artifact_path="random_forest_model",
        signature=signature,
        input_example=sample_input_clean,
        dfs_tmpdir="/Volumes/workspace/insurance_project/insurance_ml_volume/tmp"
    )

print("Model Logged With Signature")